# 04 — Visualization

Generate publication-quality figures:
1. Expression heatmap (genes × cell types)
2. Dot plot (expression + detection rate)
3. Per-cell-type volcano plots
4. Individual gene expression profiles

**Inputs:** Preprocessed data + specificity results from Notebooks 02-03  
**Outputs:** Figures in `results/heatmaps/` and `results/`

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

from src.visualization import (
    plot_expression_heatmap,
    plot_dot_plot,
    plot_gene_profile,
    plot_tau_distribution,
    plot_specificity_volcano,
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
HEATMAP_DIR = RESULTS_DIR / "heatmaps"
HEATMAP_DIR.mkdir(exist_ok=True)

## 1. Load Data

In [ ]:
mean_log = pd.read_parquet(DATA_DIR / "preprocessed_mean_log2cpm.parquet")
mean_cpm = pd.read_parquet(DATA_DIR / "preprocessed_mean_cpm.parquet")
det_df = pd.read_parquet(DATA_DIR / "preprocessed_detection_rate.parquet")
results = pd.read_csv(RESULTS_DIR / "specificity_scores.csv")

pass_only = results[results["all_pass"]]
print(f"Genes: {mean_log.shape[0]:,} | Cell types: {mean_log.shape[1]}")
print(f"Specific transcripts (all_pass): {len(pass_only):,}")

## 2. Expression Heatmap — Top Specific Genes

Show the top 3 specific genes per cell type that has hits.

In [ ]:
# Select top N genes per cell type for the heatmap
TOP_PER_CT = 3

heatmap_genes = []
for ct in pass_only["cell_type"].unique():
    ct_hits = pass_only[pass_only["cell_type"] == ct].head(TOP_PER_CT)
    heatmap_genes.extend(ct_hits["gene"].tolist())

heatmap_genes = list(dict.fromkeys(heatmap_genes))  # deduplicate, preserve order
print(f"Genes for heatmap: {len(heatmap_genes)}")

# Only include cell types that have at least one specific gene
heatmap_cell_types = pass_only["cell_type"].unique().tolist()

fig = plot_expression_heatmap(
    mean_log,
    genes=heatmap_genes,
    cell_types=heatmap_cell_types,
    title="Cell-Type Specific Transcripts — Top Hits",
    z_score=0,
    save_path=str(HEATMAP_DIR / "top_specific_genes_heatmap.png"),
)
plt.show()

## 3. Dot Plot — Selected Cell Types

In [ ]:
# Select a subset of interesting cell types for the dot plot
# (too many makes the plot unreadable)
DOT_CELL_TYPES = [
    ct for ct in [
        "hepatocyte", "T cell", "B cell", "macrophage", "neutrophil",
        "fibroblast", "endothelial cell", "epithelial cell",
        "NK cell", "monocyte", "plasma cell", "cardiomyocyte",
    ]
    if ct in mean_cpm.columns
]

# Top 2 genes per cell type for this subset
dot_genes = []
for ct in DOT_CELL_TYPES:
    ct_hits = pass_only[pass_only["cell_type"] == ct].head(2)
    dot_genes.extend(ct_hits["gene"].tolist())
dot_genes = list(dict.fromkeys(dot_genes))

if dot_genes:
    fig = plot_dot_plot(
        mean_cpm, det_df,
        genes=dot_genes,
        cell_types=DOT_CELL_TYPES,
        title="Specific Transcripts — Dot Plot",
        save_path=str(RESULTS_DIR / "dot_plot_selected.png"),
    )
    plt.show()
else:
    print("No specific genes found for the selected cell types.")

## 4. Volcano Plots — Per Cell Type

In [ ]:
# Generate volcano plots for a few cell types of interest
VOLCANO_CELL_TYPES = [ct for ct in ["hepatocyte", "T cell", "macrophage", "neuron"]
                      if ct in results["cell_type"].values]

for ct in VOLCANO_CELL_TYPES:
    fig = plot_specificity_volcano(
        results, ct,
        fc_threshold=10.0,
        det_threshold=0.25,
        label_top_n=10,
        save_path=str(RESULTS_DIR / f"volcano_{ct.replace(' ', '_')}.png"),
    )
    plt.show()

## 5. Individual Gene Profiles

In [ ]:
# Profile the top hit from each of a few cell types
PROFILE_CELL_TYPES = [ct for ct in ["hepatocyte", "T cell", "B cell", "macrophage"]
                      if ct in pass_only["cell_type"].values]

for ct in PROFILE_CELL_TYPES:
    top_gene = pass_only[pass_only["cell_type"] == ct].iloc[0]["gene"]
    fig = plot_gene_profile(
        top_gene, mean_cpm, det_df, top_n=20,
        save_path=str(RESULTS_DIR / f"profile_{top_gene}.png"),
    )
    plt.show()

## Summary

All visualizations generated. Next step: **Notebook 05 — Validation**.